> ⚠️ **历史版本归档 (Archived V0.1)**: 本实验为 IEV CloudLab V0.1 原始原型版本，仅供历史技术对照与教学基线参考。
> 🌟 **正式教学与考核**：请统一使用根目录下对应的 **IEV CloudLab V1.0 官方标准实验 (`*_pro.ipynb`)**。


# A5：Merge 匝道汇入 —— 横纵向协同控制（运动学演示）
# A5: Merge — Lateral & Longitudinal Control (Kinematic Demo)

> ⚠️ 本 notebook 为**纯 Python 运动学演示**：自车与主路车辆均为运动学模型，
> 未调用 esmini 引擎。用于理解汇入控制逻辑，真 esmini 场景见挑战任务。

## 一、实验原理 | Principle
### 🏭 工程背景 | Engineering Context

**匝道汇入（Merge）**是高速公路自动驾驶公认的难点：汇入车要在几百米内完成"找间隙→对齐速度→横向切入"三个动作，任何一个出错都是高速追尾或剐蹭。特斯拉FSD、华为ADS的实测视频中，汇入失败（被迫停车或错过出口）仍时有发生。

工程上把汇入拆成**横向（变道）+ 纵向（调速）协同**：本实验用S形轨迹做横向、PD做纵向，是教学简化版。真实系统还会考虑主线车的配合（礼让加速/减速）。

> ⚠️ **诚实声明**：本实验是**纯Python运动学演示，未调用esmini引擎**（标题已注明）。车辆用最简单的运动学模型积分，不含轮胎力、载荷转移等动力学。它的价值是讲清"横纵向协同"的**控制结构**，动力学保真度请去A1-A4的esmini实验看。

> **术语**：**间隙（Gap）**：主线上前后两车之间的空档。**S形轨迹**：横向位移随时间呈S形（先慢-后快-再慢），保证变道起止时横向速度为0，不"甩"乘客。




### 1.1 什么是匝道汇入？
车辆从匝道（加速车道）汇入主路，是自动驾驶的难点之一。需要同时解决：
- **纵向**：加速到与主路车流相当的速度，找到合适的间隙
- **横向**：平滑地从匝道变道到主路，不能急打方向

### 1.2 间隙选择（Gap Selection）
汇入前必须找到前后车都安全的"窗口"：
- 与前车距离 > 安全距离（跟驰）
- 与后车距离 > 安全距离（不被追尾）
- 实际系统中还会预测后车是否会让行

### 1.3 横纵向解耦控制
本实验用简单的解耦策略：
- 纵向：PD 跟驰（与 A2 类似）
- 横向：比例控制 `steer = k * y_err`，向目标车道靠拢

## 二、实验目的 | Objectives
1. 理解汇入场景的横纵向协同需求
2. 实现简单的变道横向控制器
3. 观察汇入轨迹的平滑性

## 三、学生任务 | Your Tasks
1. **运行**仿真，观察汇入轨迹（x-y 平面图）
2. **调参**：横向增益从 0.05 改为 0.1，观察变道是否更果断（但会不会太猛？）
3. **思考题**：如果主路车流很密（前后间距 < 10m），还应该强行汇入吗？真实系统会怎么做？（提示：等待、减速、请求让行）

## 四、验证内容 | Verification
### 🔬 试一试 | Try It

| 参数 | 你改的值 | 汇入成功？ | 横向误差 | 你的观察 |
|---|---|---|---|---|
| 横向增益 → 改大50% | | | | |
| 横向增益 → 改小50% | | | | |
| 目标车速 15 → 20 m/s | | | | |

**思考**：
- 增益过大：变道快但会冲过头（超调）吗？看横向轨迹是否越过目标车道中心线。
- 车速20 m/s：注意结果分析中的**摩擦圆警告**——这个转角在真实物理下根本不可能，运动学模型却"做到了"。这就是演示与真实仿真的区别，亲手算一算横向加速度 `v²·tan(δ)/L` 验证一下。



- [ ] 20 秒内无碰撞
- [ ] 成功到达目标车道（|y| < 0.3m）
- [ ] 轨迹平滑，无剧烈转向

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `未成功汇入` | 20秒内没到达目标车道 | 增大横向增益，或检查初始位置 |
| 轨迹振荡 | 横向增益太大 | 减小转向比例系数 |

## 五、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

| 指标 | 通过线 | 物理意义 |
|------|--------|---------|
| `merge_success` | 必须为 True（\|y\| < 0.3m） | 是否成功到达目标车道。0.3m 约为 1/6 车道宽（标准车道 3.5m），表示车辆已稳定在目标车道内 |
| `is_collision` | 必须为 False | 汇入过程中是否与主路车辆碰撞 |
| 轨迹平滑性 | 无剧烈转向 | 横向加速度应 < 2.0 m/s²，否则乘客不适且有侧翻风险（SUV） |

**Merge success criterion** (|y| < 0.3m) means the vehicle has stabilized in the target lane. 0.3m is about 1/6 of a standard 3.5m lane — tight enough to ensure the vehicle is truly in-lane, not straddling the line.

### 5.2 横纵向解耦控制的合理性 | Is Decoupled Control Reasonable?

本实验用最简单的解耦策略：
- 纵向：PD 跟驰（`a = 0.8·gap_err + 0.5·v_err`，限幅 [-3.0, 2.0]）
- 横向：比例控制（`steer = 0.05·y_err`，限幅 ±0.1 rad）

**合理性**：
- ✅ 教学上清晰：学生能分别看到纵向和横向的作用
- ✅ 低速汇入时耦合效应弱，解耦近似成立
- ⚠️ 高速或大曲率时，纵横向强耦合（轮胎摩擦圆限制），解耦控制会失效——这是进阶内容（MPC 统一优化，见 E3）

**横向增益 0.05 的整定**：
- 增益太小（如 0.01）：变道太慢，20 秒内到不了目标车道 → `merge_success=False`
- 增益太大（如 0.2）：转向过猛，轨迹振荡，甚至超出 ±0.1 rad 限幅进入饱和
- 0.05 是试出来的 sweet spot：约 8~10 秒完成变道，无振荡

### 5.3 轨迹曲线特征说明 | Reading the Trajectory

成功的 x-y 轨迹应呈 **S 形（S-curve）**：
1. **起始段**：y 从 -3.5m（匝道）开始，曲率逐渐增大
2. **中段**：曲率最大，横向速度最快
3. **收敛段**：y 渐近 0m，曲率减小，平稳入道

如果轨迹呈 **直线**（直接斜切）：横向增益过大，或限幅太松。
如果轨迹**振荡**（y 在 0 上下来回）：增益过大产生超调，需要减小或加微分项（PD）。
如果 20 秒后 y 还在 -1m 附近：增益太小，加大到 0.08 试试。

### 5.4 结果合理性判断 | Is the Result Reasonable?

- **8~12 秒汇入，无碰撞，轨迹 S 形平滑**：优秀。
- **18~20 秒才汇入（压线通过）**：临界。真实高速汇入时，加速车道长度有限（通常 200~300m），太慢会被迫停车等待。
- **汇入成功但与后车距离很近**：纵向控制需改进。本实验的纵向只跟前车，没考虑后车。真实汇入需要同时保证前后间隙（Gap Selection，见原理 1.2）。
- **碰撞**：通常是纵向没跟好（前车减速时没及时减速），或横向切入时与旁车道车辆侧碰。检查 PD 参数和转向限幅。

### 5.5 误差来源与简化假设 | Limitations

> 本 notebook 为**纯 Python 运动学演示**，以下简化必须在分析结果时牢记：

1. **无感知噪声**：`observation` 中的目标车位置是真值。真实系统有感知误差（±0.3m）和延迟（~100ms），汇入决策需要鲁棒性设计。
2. **主路车辆不交互**：主路车按预设运动，不会让行也不会加速逼你。真实汇入是**博弈**——后车可能加速不让你（挑战任务 3 的思考题）。
3. **运动学模型**：自行车模型，无侧倾、无轮胎饱和。`steer=0.1 rad` 在 20m/s 下的横向加速度约为 $v²·δ/L ≈ 14$ m/s²——**这在物理上不可能**（轮胎极限约 8 m/s²）！本实验速度低，问题不暴露；高速场景必须用动力学模型。
4. **解耦假设**：纵横向独立设计。真实车辆受摩擦圆约束：$\sqrt{a_x² + a_y²} ≤ μg$，纵向急加速时横向能力下降。

### 5.6 与真实汇入的对比 | vs. Real-World Merging

| 真实汇入要素 | 本实验 | 差距 |
|------------|--------|------|
| 间隙选择（找前后车窗口） | 固定场景，无选择逻辑 | 需要加决策层 |
| 后车博弈（对方让不让） | 后车无反应 | 需要意图预测/博弈论 |
| 感知不确定性 | 真值输入 | 需要加噪声鲁棒性 |
| 动力学约束（摩擦圆） | 运动学，无约束 | 高速需动力学模型 |
| 舒适性（横向加速度<2m/s²） | 未断言 | 可加 `max_lat_a` 指标 |

**工程意义**：本实验的价值在于让学生**第一次**把横向和纵向控制放在同一个闭环里跑通。真实系统的复杂度是 10 倍，但基本框架（感知→决策→横纵向控制）是一样的。理解了解耦控制的局限，才能理解为什么需要 MPC（E3）。


### 5.6 设计改进思考 | Design Improvement

运动学演示的局限，正是工程师的改进清单：

1. **换动力学模型**：把运动学自行车换成带轮胎Pacejka模型的动力学模型，转角指令先过"摩擦圆"检查：`a_y = v²·tan(δ)/L ≤ μg`。本实验在20 m/s+大转角下a_y远超物理极限，真实车会直接甩尾。

2. **间隙选择策略**：本实验假设间隙已选定。真实系统要先做间隙评估：前间隙、后间隙哪个更安全？主线后车是否在加速封堵？这需要预测模型（见E3的MPC思想）。

3. **失败回退**：汇入失败怎么办？真实系统有Plan B：减速等待下一个间隙，或放弃本次汇入走下一个出口。演示只跑成功案例，工程必须设计失败路径——**功能安全的思维方式**。

> **一句话总结**：演示告诉你"应该怎么做"，工程还要回答"做不到时怎么办"。



In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import KinematicBicycleModel
print('✓ 导入成功（纯运动学模型，未使用 esmini 引擎）')


## 1. 汇入控制器（横纵向）

In [ ]:
class MergeController(BaseController):
    """匝道汇入：纵向跟驰 + 横向变道"""
    def __init__(self, config=None):
        super().__init__(config)
        self.target_y = 0.0  # 目标车道
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        # 纵向：跟随前车
        if targets:
            lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
            gap_err = (abs(lead['x'] - ego['x']) - 4.5) - ego['v'] * 1.0
            a = max(-3.0, min(2.0, 0.8 * gap_err + 0.5 * (lead['v'] - ego['v'])))
        else:
            a = 0.5
        # 横向：向目标车道靠拢（简化：固定转向角）
        y_err = self.target_y - ego['y']
        steer = max(-0.1, min(0.1, 0.05 * y_err))
        return {'target_a': a, 'steer_angle': steer}
print('✓ Merge 控制器就绪')


## 2. 运动学仿真（纯 Python，未使用 esmini）

In [ ]:
# 汇入场景：自车从 y=-3 汇入主路 y=0
controller = MergeController()
ego = KinematicBicycleModel()
ego.reset(x=10.0, y=-3.0, yaw=0.0, v=12.0)
# 主路车辆：硬编码运动学函数（演示用，非 esmini 场景）
import math
def main_road_vehicle(t, x0=40, v=14.0):
    return {'x': x0 + v*t, 'y': 0.0, 'v': v}

is_collision, merge_success = False, False
history = []
for step in range(400):
    t = step * 0.05
    lead = main_road_vehicle(t)
    obs = {'ego': {'x': ego.x, 'y': ego.y, 'yaw': ego.yaw, 'v': ego.v},
           'targets': [lead], 'sim_time': t, 'dt': 0.05}
    act = controller.step(obs)
    ego.step(act['target_a'], act['steer_angle'], 0.05)
    history.append((t, ego.x, ego.y, ego.v))
    # 碰撞检测
    if abs(lead['x'] - ego.x) < 4.5 and abs(lead['y'] - ego.y) < 1.5:
        is_collision = True
        break
    # 汇入成功：到达目标车道
    if abs(ego.y) < 0.3 and t > 5:
        merge_success = True

print(f'✓ 完成：{len(history)} 步，碰撞={is_collision}，汇入成功={merge_success}')


## 3. 评测断言

In [ ]:
assert not is_collision, '发生碰撞!'
assert merge_success, '未成功汇入!'
print('✅ 评测通过！')


## 4. 可视化

In [ ]:
import matplotlib.pyplot as plt
times = [t for t, x, y, v in history]
xs = [x for t, x, y, v in history]
ys = [y for t, x, y, v in history]
plt.figure(figsize=(12, 4))
plt.plot(xs, ys, label='Ego trajectory')
plt.axhline(y=0, color='gray', linestyle='--', label='Main road')
plt.axhline(y=-3, color='gray', linestyle=':', label='Ramp')
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('Merge Trajectory (Kinematic Demo)')
plt.annotate('S-curve: smooth lane change', xy=(xs[len(xs)//2], ys[len(ys)//2]), fontsize=8,
             xytext=(10, 20), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='blue', alpha=0.5))
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()


**图说明**：S形汇入轨迹——从匝道（下方虚线）平滑切入主路（上方虚线）。注意这条漂亮的曲线是**运动学**算出来的：它假设轮胎有无限抓地力。试一试里让你算横向加速度，就是要亲手戳破这个假设——真实物理下这条曲线可能根本走不出来。


## 六、🏆 挑战任务
1. 将本演示迁移到 esmini 真闭环（需 xosc 汇入场景）。
2. 加入间隙选择逻辑：选择前后车都安全的时机汇入。


---
## 阶段五：结果文件归档与平台官方评测 | Auto-Grading & Results Sync


In [ ]:
# 阶段五：结果文件归档与平台官方评测
import json, os
from common.evaluator import grade_metrics

record = {
    'experiment': 'A5_merge_kinematic_demo',
    'metrics': {
        'completed': bool(merge_success and not is_collision)
    }
}
saved_ok = False
for target_dir in ['/home/jovyan/labs/data/results', 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'A5_merge_kinematic_demo.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f'✓ 结果文件已成功归档至: {out_file}')
        saved_ok = True
        break
    except OSError:
        continue
eval_report = grade_metrics('A5_merge_kinematic_demo', record['metrics'])
print(f'🏆 官方评分: {eval_report["score"]} / 100 分 ({'✅ 通过' if eval_report["passed"] else '❌ 未通过'})')


---

## 下一个实验 | Next →

**[B1_idm_calibration](./B1_idm_calibration.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
